# Week 43, Day 3: Monotonic stacks

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w043d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
temps = [13, 12, 15, 11, 9, 12, 16]
wait = [0] * len(temps)
stack = []                                   # indexes of days still waiting for a warmer one
for day, t in enumerate(temps):
    while stack and temps[stack[-1]] < t:
        earlier = stack.pop()
        wait[earlier] = day - earlier
    stack.append(day)
print(wait, stack)

In [ ]:
def next_greater(nums):
    answer = [-1] * len(nums)
    stack = []                                 # indexes, with values decreasing from bottom to top
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x            # x is the first bigger item to the right of that one
        stack.append(i)
    return answer

print(next_greater([2, 1, 2, 4, 3]))
print(next_greater([5, 4, 3, 2, 1]), next_greater([1, 2, 3]))

In [ ]:
def stock_span(prices):
    """For each day, how many days in a row up to and including it had a price at most today's."""
    spans = []
    stack = []                                   # indexes of earlier days with higher prices
    for i, p in enumerate(prices):
        while stack and prices[stack[-1]] <= p:
            stack.pop()
        spans.append(i - stack[-1] if stack else i + 1)   # the previous greater day, or none
        stack.append(i)
    return spans

print(stock_span([100, 80, 60, 70, 60, 75, 85]))

In [ ]:
def largest_rectangle(heights):
    best = 0
    stack = []                                       # indexes, heights increasing upwards
    for i, h in enumerate(heights + [0]):            # a final 0 pops everything left
        while stack and heights[stack[-1]] > h:
            top = stack.pop()
            left = stack[-1] if stack else -1        # the previous smaller bar
            width = i - left - 1                     # i is the next smaller bar
            best = max(best, heights[top] * width)
        stack.append(i)
    return best

print(largest_rectangle([2, 1, 5, 6, 2, 3]), largest_rectangle([4, 4, 4]), largest_rectangle([]))

In [ ]:
from collections import deque

def window_max(nums, k):
    out = []
    window = deque()                                  # indexes; their values decrease front to back
    for i, x in enumerate(nums):
        while window and nums[window[-1]] <= x:
            window.pop()
        window.append(i)
        if window[0] <= i - k:
            window.popleft()                          # slid out of the window
        if i >= k - 1:
            out.append(nums[window[0]])
    return out

print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

> **Key idea.** A monotonic stack keeps its items in increasing or decreasing order: each new item first pops every item it settles. For the next greater item, pop while the top is smaller (the new item is their answer); for the previous greater, pop while the top isn't bigger, and the top left over is the new item's answer; smaller versions flip the comparison. Store indexes, and decide carefully between `<` and `<=` for equal values. Every index is pushed and popped once, so it's O(n). The largest rectangle in a histogram uses an increasing stack and a final 0 as a sentinel; a monotonic deque gives every sliding window's maximum in O(n).

### Exercise: The next greater value

Write `next_greater(nums)`: for each item, the first value to its right that is **strictly** bigger, or `-1` if there's none. One pass with a stack of indexes, O(n); the checks use 100,000 numbers.

In [ ]:
def next_greater(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(nums):\n    return [next((y for y in nums[i + 1:] if y > x), -1) for i, x in enumerate(nums)]\ndef test_small():\n    \"small\"\n    assert next_greater([2, 1, 2, 4, 3]) == [4, 2, 4, -1, -1] and next_greater([]) == [] and next_greater([3, 3]) == [-1, -1]\n    rng = random.Random(1)\n    for _ in range(200):\n        nums = [rng.randint(0, 5) for _ in range(rng.randint(0, 12))]\n        assert next_greater(nums) == slow(nums), nums\ndef test_big():\n    \"100,000\"\n    nums = list(range(100_000, 0, -1)) + [10 ** 6]\n    assert next_greater(nums)[:3] == [10 ** 6] * 3")

### Exercise: Days until warmer

Write `days_until_warmer(temps)`: for each day, how many days later the first **strictly** warmer day comes, or 0 if none does. One pass with a monotonic stack.

In [ ]:
def days_until_warmer(temps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(t):\n    return [next((j - i for j in range(i + 1, len(t)) if t[j] > t[i]), 0) for i in range(len(t))]\ndef test_warmer():\n    \"warmer days\"\n    assert days_until_warmer([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0] and days_until_warmer([30, 30, 31]) == [2, 1, 0]\n    rng = random.Random(2)\n    for _ in range(200):\n        t = [rng.randint(10, 15) for _ in range(rng.randint(0, 12))]\n        assert days_until_warmer(t) == slow(t), t")

### Exercise: The previous smaller value

Write `previous_smaller(nums)`: for each item, the **index** of the nearest item to its left that is strictly smaller, or `-1`. Loop left to right; pop while the top isn't smaller, then the top (if any) is the answer for the new item.

In [ ]:
def previous_smaller(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(nums):\n    return [next((j for j in range(i - 1, -1, -1) if nums[j] < nums[i]), -1) for i in range(len(nums))]\ndef test_prev():\n    \"previous smaller\"\n    assert previous_smaller([4, 5, 2, 10, 8]) == [-1, 0, -1, 2, 2] and previous_smaller([1, 1, 1]) == [-1, -1, -1]\n    rng = random.Random(3)\n    for _ in range(200):\n        nums = [rng.randint(0, 5) for _ in range(rng.randint(0, 12))]\n        assert previous_smaller(nums) == slow(nums), nums")

### Exercise: The stock span

Write `StockSpanner` with `next(price)`, called once per day: return today's **span**, the number of consecutive days up to and including today on which the price was at most today's. Each call must be O(1) amortised: keep a stack of `(price, span)` pairs, and when today's price is at least the top's, pop it and add its span to today's.

In [ ]:
class StockSpanner:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_span():\n    \"spans\"\n    s = StockSpanner()\n    assert [s.next(p) for p in [100, 80, 60, 70, 60, 75, 85]] == [1, 1, 1, 2, 1, 4, 6]\n    rng = random.Random(4)\n    prices = [rng.randint(1, 20) for _ in range(300)]\n    s = StockSpanner()\n    for i, p in enumerate(prices):\n        want = 0\n        while i - want >= 0 and prices[i - want] <= p:\n            want += 1\n        assert s.next(p) == want\ndef test_fast():\n    \"many days\"\n    s = StockSpanner()\n    for p in range(100_000):\n        span = s.next(p)\n    assert span == 100_000")

### Exercise: Sliding window maximum

Write `window_max(nums, k)`: the maximum of every window of `k` neighbouring items, in order, with a monotonic `deque` of indexes, in O(n). `k` must be from 1 to `len(nums)`, else `ValueError`. The checks use 200,000 numbers with `k = 1000`, far too slow for `max` of each window.

In [ ]:
from collections import deque

def window_max(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_small():\n    \"small\"\n    assert window_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7] and window_max([4], 1) == [4] and window_max([2, 2, 2], 2) == [2, 2]\n    rng = random.Random(5)\n    for _ in range(200):\n        nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 12))]\n        k = rng.randint(1, len(nums))\n        assert window_max(nums, k) == [max(nums[i:i + k]) for i in range(len(nums) - k + 1)], (nums, k)\n    for bad in [0, 4]:\n        try:\n            window_max([1, 2, 3], bad)\n            assert False\n        except ValueError:\n            pass\ndef test_big():\n    \"200,000 numbers\"\n    rng = random.Random(6)\n    nums = [rng.randint(0, 10 ** 6) for _ in range(200_000)]\n    got = window_max(nums, 1000)\n    assert len(got) == 199_001 and got[0] == max(nums[:1000]) and got[-1] == max(nums[-1000:])")

### Exercise: The largest rectangle

Write `largest_rectangle(heights)`: the area of the largest rectangle under a histogram whose bars have width 1 and the given heights. Use an increasing stack of indexes and a final `0` sentinel, O(n); the checks use 100,000 bars.

In [ ]:
def largest_rectangle(heights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(h):\n    return max((min(h[i:j]) * (j - i) for i in range(len(h)) for j in range(i + 1, len(h) + 1)), default=0)\ndef test_small():\n    \"small\"\n    assert largest_rectangle([2, 1, 5, 6, 2, 3]) == 10 and largest_rectangle([2, 4]) == 4 and largest_rectangle([]) == 0 and largest_rectangle([5]) == 5\n    rng = random.Random(7)\n    for _ in range(200):\n        h = [rng.randint(0, 6) for _ in range(rng.randint(0, 10))]\n        assert largest_rectangle(h) == slow(h), h\ndef test_big():\n    \"100,000 bars\"\n    assert largest_rectangle(list(range(1, 100_001))) == 2_500_050_000 and largest_rectangle([7] * 100_000) == 700_000")

### Bug hunt: Equal isn't greater

`next_greater` should find the next **strictly** bigger value, but it pops while the top is **at most** the new item, so an equal value answers it: this prints `[2, 3, 3, -1]`. Fix the comparison so it prints `[3, 3, -1, -1]`.

In [ ]:
def next_greater(nums):
    answer = [-1] * len(nums)
    stack = []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] <= x:
            answer[stack.pop()] = x
        stack.append(i)
    return answer

print(next_greater([2, 2, 3, 1]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"strictly greater\"\n    expect_output(\"[3, 3, -1, -1]\")")

### Bug hunt: Values can't count days

This stack holds **temperatures** instead of day indexes, so when a warmer day arrives it can't tell which day it's answering or how far back it was: it treats a temperature as a day number, and this prints `[0, 0, 0, 0, 0]`. Store indexes, so it prints `[2, 1, 0, 0, 0]`.

In [ ]:
def days_until_warmer(temps):
    wait = [0] * len(temps)
    stack = []
    for day, t in enumerate(temps):
        while stack and stack[-1] < t:
            earlier = stack.pop()
            if earlier < len(wait):
                wait[earlier] = day - earlier
        stack.append(t)
    return wait

print(days_until_warmer([20, 19, 21, 18, 17]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"indexes on the stack\"\n    expect_output(\"[2, 1, 0, 0, 0]\")")

### Bug hunt: A maximum that has left

This window-maximum deque never removes indexes that have slid out of the window, so an old big value keeps being reported: this prints `[9, 9, 9, 9]`. Drop the front index when it's out of the window, so it prints `[9, 4, 4, 5]`.

In [ ]:
from collections import deque

def window_max(nums, k):
    out = []
    window = deque()
    for i, x in enumerate(nums):
        while window and nums[window[-1]] <= x:
            window.pop()
        window.append(i)
        if i >= k - 1:
            out.append(nums[window[0]])
    return out

print(window_max([9, 1, 4, 2, 5], 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"drop what's left the window\"\n    expect_output(\"[9, 4, 4, 5]\")")

### Puzzle: Next smaller to the right

Arrange the lines into a function giving, for each item, the next strictly smaller value to its right (or -1). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
nums = [3, 7, 1, 7, 2]
stack, pops = [], 0
for i, x in enumerate(nums):
    while stack and nums[stack[-1]] < x:
        stack.pop()
        pops += 1
    stack.append(i)

p = ___   # stack at the end
q = ___   # pops
r = ___   # the most pushes a monotonic stack can make for a list of 5 items
s = ___   # the order of values from bottom to top in a next-greater stack: "increasing" or "decreasing"
t = ___   # the cost of window_max with a monotonic deque: "O(n)" or "O(nk)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 3, 4] and q == 2 and r == 5, \"p, q and r\"\n    assert s == \"decreasing\" and t == \"O(n)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the largest rectangle of ones

`grid` is a list of equal-length strings of `"1"` and `"0"`. Write `largest_block(grid)`: the area of the largest rectangle made only of `"1"`s. Treat each row as the floor of a histogram: the height of column `c` is how many `"1"`s in a row end at that row in that column (0 if this row's square is `"0"`). Update the heights row by row and run the largest-rectangle method on each, for O(rows × columns) in all. The checks use a 200 by 200 grid.

In [ ]:
def largest_block(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(grid):\n    rows, cols = len(grid), len(grid[0]) if grid else 0\n    best = 0\n    for r1 in range(rows):\n        for r2 in range(r1, rows):\n            for c1 in range(cols):\n                for c2 in range(c1, cols):\n                    if all(grid[r][c] == \"1\" for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)):\n                        best = max(best, (r2 - r1 + 1) * (c2 - c1 + 1))\n    return best\ndef test_small():\n    \"small\"\n    assert largest_block([\"10100\", \"10111\", \"11111\", \"10010\"]) == 6 and largest_block([\"0\"]) == 0 and largest_block([\"1\"]) == 1 and largest_block([]) == 0\n    rng = random.Random(8)\n    for _ in range(100):\n        cols = rng.randint(1, 5)\n        g = [\"\".join(rng.choice(\"110\") for _ in range(cols)) for _ in range(rng.randint(1, 5))]\n        assert largest_block(g) == slow(g), g\ndef test_big():\n    \"200 by 200\"\n    g = [\"1\" * 200] * 200\n    assert largest_block(g) == 40_000\n    rng = random.Random(9)\n    g = [\"\".join(rng.choice(\"1111110\") for _ in range(200)) for _ in range(200)]\n    assert largest_block(g) > 0")

## Warm-up

For each day, how many days until a warmer one? The stack holds the days still waiting, and their temperatures **decrease** from bottom to top. Each new day answers every waiting day colder than itself, popping them, then waits itself. The day with 16 never finds a warmer one, so it's still on the stack at the end with a 0.

## The next greater item

The question: for each item, what's the first item to its **right** that's bigger? The brute-force answer checks every later item, O(n²). The stack answer is one pass:

The reasoning, as a story: items wait on the stack until someone bigger arrives. An item can't be answered by anyone it's bigger than, so the waiting items are always in **decreasing** order: the stack is **monotonic** (always moving one way). When a new item arrives, it settles every waiting item smaller than it, from the top down, then joins the wait.

Store **indexes** on the stack, not values: the index lets you write the answer into the right place, and work out distances (as with the days until warmer).

## Four variations

Two choices make four versions. Which way are you looking, and for bigger or smaller?

| Question | Loop | Pop while the top is... | Stack order (bottom to top) |
|---|---|---|---|
| next **greater** to the right | left to right | **smaller** than the new item; the new item answers it | decreasing |
| next **smaller** to the right | left to right | **bigger** than the new item | increasing |
| previous **greater** to the left | left to right | **not bigger** (smaller or equal); after popping, the top is the answer for the **new** item | decreasing |
| previous **smaller** to the left | left to right | **not smaller**; the top after popping answers the new item | increasing |

For the "previous" versions, the answer belongs to the new item, not the popped ones: after popping everything that can't be the answer, whatever is left on top is the nearest one that can. **Equal values** need a decision: should an equal item count as "greater"? The choice between `<` and `<=` in the `while` decides, and getting it wrong is the commonest bug.

## Why it's O(n)

A `while` inside a `for` looks like O(n²). But every index is **pushed once** and **popped at most once**, over the whole run. So the inner loop runs at most n times **in total**, not n times per item: O(n) altogether. It's the same amortised argument as Week 41's sliding window: one item may pop many, but every pop uses up an item that can never be popped again.

## The largest rectangle

A histogram is a row of bars of width 1 and different heights. What's the largest rectangle that fits under the bars? For each bar, the widest rectangle **as tall as that bar** stretches left and right until it meets a shorter bar: so each bar needs its previous smaller and next smaller bars. An increasing stack finds both at once: when a bar is popped by a shorter one, the shorter one is its next smaller, and the bar beneath it on the stack is its previous smaller.

The extra `0` at the end is a **sentinel** (Week 42's idea in another form): it's shorter than every bar, so it settles everything still waiting, with no separate clean-up loop.

## A monotonic deque

Week 41 slid a window along a list, keeping its total. The **maximum** of each window is harder: when the biggest item leaves the window, what's the next biggest? A **monotonic deque** keeps the candidates: indexes whose values **decrease** from front to back.

- When a new item arrives, pop from the **back** every index with a value no bigger than it: they can never be a window's maximum again, because the new item is bigger and will stay in the window longer.
- Pop from the **front** any index that has slid out of the window.
- The **front** is then the current window's maximum.

Each index enters and leaves the deque once, so it's O(n) in all, however big `k` is; recomputing `max` of each window would be O(nk).

## Practice

## Quiz

## Daily challenge

## Before you go

For each of the four variations, write the `while` condition and say whose answer gets written when you pop. Then predict what changes if you swap `<` for `<=`. Tomorrow: brackets and undo, where stacks parse and evaluate whole expressions, and undo and redo get a proper design.